# 🚀 Two-Agent News Sentiment Analyzer for AAPL

This notebook implements a dynamic RAG pipeline for financial sentiment analysis using a two-agent FinRobot workflow:
1. Pulls the latest news articles from 11 sources using the custom multi-provider aggregator.
2. For each article, queries a local LangChain FAISS vector store to retrieve semantic calibration examples.
3. Instantiates a **Sentiment Scorer Agent** (loaded from `sentiment_prompt.txt`) to analyze each article against its few-shot anchors and output individual scores, confidences, and risk factors.
4. Instantiates a **Senior Sentiment Analyst (CIO) Agent** (loaded from `cio_prompt.txt`) to review the scores, compute the confidence-weighted average sentiment, flag warnings, and compile the final JSON report matching the target schema.

## 1. Load Environment & Setup Vector Store (FAISS + NVIDIA Embeddings)

In [ ]:
import sys
import os
from dotenv import load_dotenv

# Ensure the current directory is in the python path for importing modules
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

sentiment_dir = os.path.dirname(notebook_dir)
if sentiment_dir not in sys.path:
    sys.path.insert(0, sentiment_dir)

project_root = os.path.dirname(sentiment_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Load environment variables from .env.local
load_dotenv("../.env.local")

In [ ]:
import pandas as pd
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Read and clean environment variables
nvidia_embedding_model = os.getenv("NVIDIA_EMBEDDING_MODEL", "nvidia/nv-embed-v1").strip('"\' ')
nvidia_base_model = os.getenv("NVIDIA_BASE_MODEL", "").strip('"\' ')
nvidia_api_endpoint = os.getenv("NVIDIA_API_ENDPOINT", "https://integrate.api.nvidia.com/v1").strip('"\' ')
nvidia_api_key = os.getenv("NVIDIA_API_KEY", "").strip('"\' ')

print(f"Initializing NVIDIA Embeddings wrapper ({nvidia_embedding_model})...")
embeddings = NVIDIAEmbeddings(
    model=nvidia_embedding_model,
    nvidia_api_key=nvidia_api_key,
    base_url=nvidia_api_endpoint
)

In [ ]:
# Load historical sentiment dataset
csv_path = "../data/financial_sentiment.csv"
print(f"Loading dataset: {csv_path}...")
df_sentiment = pd.read_csv(csv_path)
df_sentiment = df_sentiment.dropna(subset=["Sentence", "Sentiment"])

# Indexing a subset of 300 rows for fast database creation and query speed
limit_rows = 300
df_subset = df_sentiment.head(limit_rows)
print(f"Indexing {len(df_subset)} records into FAISS vector database...")

documents = [
    Document(page_content=row["Sentence"], metadata={"sentiment": row["Sentiment"]})
    for _, row in df_subset.iterrows()
]

# Build local FAISS database
db = FAISS.from_documents(documents, embeddings)
print("[+] FAISS Local Vector Store created successfully!")

## 2. Initialize Two-Agent FinRobot Workflow & Prepare Data

In [ ]:
from functions import llm_config, base_llm_config


In [ ]:
ticker = "AAPL"
news_limit = 5  # Score top 5 articles

In [ ]:
import json
import datetime
import autogen
from finrobot.agents.workflow import FinRobot
from autogen import UserProxyAgent
from sentiment.functions.aggregator.aggregator import fetch_aggregate_all_news

# 1. Prepare articles data with matching calibration examples in Python for efficiency

# Fetch news feed
print(f"Fetching consolidated news feed for {ticker}...")
df_news = fetch_aggregate_all_news(symbol=ticker, limit=100)

if df_news.empty:
    raise ValueError(f"No news articles found for symbol {ticker}.")

df_news_limited = df_news.head(news_limit)



articles_to_analyze = []
for idx, row in df_news_limited.iterrows():
    title = row.get('title', 'No Title')
    source = row.get('source', 'Unknown Source')
    date = str(row.get('date', 'Unknown Date'))
    summary = row.get('summary', '')
    text = row.get('text', '')
    
    is_summary_missing = not summary or pd.isna(summary) or not isinstance(summary, str) or len(summary.strip()) < 5
    is_text_missing = not text or pd.isna(text) or not isinstance(text, str) or len(text.strip()) < 5
    if is_summary_missing:
        if not is_text_missing:
            summary = f"{title}\n\n{text}"
        else:
            summary = title
            
    if not summary or pd.isna(summary) or not isinstance(summary, str) or len(summary.strip()) < 5:
        # Insufficient data placeholder
        articles_to_analyze.append({
            "title": title,
            "source": source,
            "published_at": date,
            "summary": "Insufficient data (missing summary).",
            "calibration_examples": []
        })
        continue

    # Query vector store
    similar_docs = db.similarity_search(summary, k=2)
    calibration_examples = [
        {"sentence": doc.page_content, "sentiment": doc.metadata["sentiment"]}
        for doc in similar_docs
    ]
    
    articles_to_analyze.append({
        "title": title,
        "source": source,
        "published_at": date,
        "summary": summary,
        "calibration_examples": calibration_examples
    })



In [ ]:
# 2. Instantiate the UserProxyAgent
user_proxy = UserProxyAgent(
    name="User_Proxy",
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "") and "TERMINATE" in x.get("content", ""),
    max_consecutive_auto_reply=5,
    code_execution_config={"use_docker": False}
)

In [ ]:
with open("../schema_json/scorer_schema.json", "r") as f:
    schema_str = f.read()

with open("../prompts/sentiment_prompt.txt", "r") as f:
    scorer_prompt_template = f.read()
    
# 3. Initialize the Sentiment Scorer Agent from external prompt txt
scorer_profile = scorer_prompt_template.format(
    SCHEMA=schema_str,
    EXAMPLES="Use the matching 'calibration_examples' list provided inline inside the user message for each article."
)

scorer_agent = FinRobot(
    agent_config={
        "name": "Sentiment_Scorer",
        "description": "Scoration specialist agent.",
        "profile": scorer_profile,
        "toolkits": []
    },
    llm_config=llm_config
)

In [ ]:
# Load the raw prompt template
with open("../prompts/cio_prompt.txt", "r") as f:
    cio_prompt_template = f.read()

# Load the target output schema description (schema_str)
with open("../schema_json/sentiment_schema.json", "r") as f:
    schema_str = f.read()

# Load the target output example JSON
with open("../schema_json/cio_output_schema.json", "r") as f:
    output_str = f.read()

# Load the scored articles input example JSON
with open("../schema_json/cio_scored_articles.json", "r") as f:
    scored_articles_str = f.read()

# 4. Initialize the Senior Sentiment Analyst (CIO) Agent
cio_profile = cio_prompt_template.format(
    SCHEMA=schema_str,
    EXAMPLES=scored_articles_str,
    OUTPUT=output_str
)

cio_agent = FinRobot(
    agent_config={
        "name": "Senior_Sentiment_Analyst",
        "description": "Consolidation and aggregation agent.",
        "profile": cio_profile,
        "toolkits": []
    },
    llm_config=base_llm_config
)


## 3. Execute Conversational Analysis & Generate Report

In [ ]:
# 5. Run the sequential two-agent workflow
print("\n[+] Step 1: Sentiment Scorer analyzing articles...")
scorer_task = (
    "Please score the following articles according to your instructions:\n\n"
    f"{json.dumps(articles_to_analyze, indent=2)}\n\n"
    "Respond with the list of scored articles."
)
user_proxy.initiate_chat(scorer_agent, message=scorer_task)

# Extract scorer output
scorer_msg = user_proxy.last_message(scorer_agent).get("content", "")
if scorer_msg.endswith("TERMINATE"):
    scorer_msg = scorer_msg[:-9].strip()

In [ ]:
print(scorer_msg)

In [ ]:
# 6. Pass scored articles to the Senior Sentiment Analyst for consolidation
print("\n[+] Step 2: Senior Sentiment Analyst consolidating scores...")
cio_task = (
    "Please consolidate the following scored articles into the final JSON report according to your instructions:\n\n"
    f"{scorer_msg}\n\n"
    "Generate the final JSON block matching the schema."
)
user_proxy.initiate_chat(cio_agent, message=cio_task)

# Extract CIO output
final_report_msg = user_proxy.last_message(cio_agent).get("content", "")
if final_report_msg.endswith("TERMINATE"):
    final_report_msg = final_report_msg[:-9].strip()

# Clean Markdown JSON syntax wrapper if present
if "```json" in final_report_msg:
    final_report_msg = final_report_msg.split("```json")[1].split("```")[0].strip()
elif "```" in final_report_msg:
    final_report_msg = final_report_msg.split("```")[1].split("```")[0].strip()

print("\n================ FINAL REPORT ================")
try:
    final_report = json.loads(final_report_msg)
    print(json.dumps(final_report, indent=2))
except Exception as e:
    print(f"Error parsing final report: {e}")
    print("Raw Output:")
    print(final_report_msg)

In [ ]:
print(final_report_msg)